In [ ]:
# Install: pip install datasets openai pandas pillow
from datasets import load_dataset
import pandas as pd
from pathlib import Path
import base64
from openai import OpenAI
import os

# --- 1. SETUP ---
# Initialize OpenAI client (Ensure OPENAI_API_KEY is in your environment variables)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# --- 2. DATA LOADING ---
print("Loading product dataset...")
try:
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:10]")
    products_df = pd.DataFrame(dataset)
    # Ensure local path column exists for processing
    products_df['image_path'] = products_df['filename'].apply(lambda x: f"images/{x}") 
except Exception as e:
    print(f"⚠ Could not load dataset: {e}")
    products_df = pd.DataFrame([{"id": 1, "name": "Wireless Headphones", "price": 79.99, "image_path": "images/product1.jpg"}])

# --- 3. VISION INTEGRATION FUNCTIONS ---
def encode_image(image_path):
    """Encodes a local image to base64 for the API."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

def generate_product_description(product_row):
    """Uses GPT-4 Vision to generate a compelling product description."""
    try:
        base64_image = encode_image(product_row['image_path'])
        
        prompt = f"Write a compelling, SEO-friendly product listing description for a product named '{product_row.get('productDisplayName', 'this item')}'. It costs {product_row.get('price', 'N/A')}. Focus on features and benefits."

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a professional e-commerce copywriter."},
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
                ]}
            ],
            max_tokens=300
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error generating description: {e}"

# --- 4. EXECUTION ---
print("Generating descriptions...")
# Only run on the first few to save API credits
products_df['description'] = products_df.head(5).apply(generate_product_description, axis=1)

print("\n✓ Processing Complete!")
print(products_df[['productDisplayName', 'description']].head())